In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("netflix_titles.csv")
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df.shape

(8807, 12)

In [4]:
df.columns.tolist()

['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


In [6]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df = df.drop_duplicates()
df.shape

(8807, 12)

In [9]:
df.fillna({
    "director": "Unknown",
    "cast": "Unknown",
    "country": "Unknown",
    "rating": "Not Rated",
    "duration": "Unknown",
    "listed_in": "Unknown",
    "description": "No Description"
}, inplace=True)

In [10]:
df = df.dropna(subset=["date_added"])
df.shape

(8797, 12)

In [11]:
df["date_added"] = pd.to_datetime(df["date_added"], errors="coerce")

In [12]:
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month
df["month_name"] = df["date_added"].dt.strftime("%b")

In [13]:
df = df.dropna(subset=["date_added", "year_added", "month_added"])

In [14]:
df[["date_added", "year_added", "month_added", "month_name"]].head()

,date_added,year_added,month_added,month_name
0,2021-09-25,2021.0,9.0,Sep
1,2021-09-24,2021.0,9.0,Sep
2,2021-09-24,2021.0,9.0,Sep
3,2021-09-24,2021.0,9.0,Sep
4,2021-09-24,2021.0,9.0,Sep


In [15]:
df["year_added"] = df["year_added"].astype(int)
df["month_added"] = df["month_added"].astype(int)

In [16]:
total_titles = df["show_id"].nunique()

In [17]:
total_movies = (df["type"] == "Movie").sum()

In [18]:
total_tvshows = (df["type"] == "TV Show").sum()

In [19]:
top_genre = df["listed_in"].mode()[0]

In [20]:
df["combined"] = (
    df["type"].fillna("").astype(str) + " " +
    df["listed_in"].fillna("").astype(str) + " " +
    df["description"].fillna("").astype(str)
)

In [21]:
df[["title", "combined"]].head()

,title,combined
0,Dick Johnson Is Dead,Movie Documentaries As her father nears the en...
1,Blood & Water,"TV Show International TV Shows, TV Dramas, TV ..."
2,Ganglands,"TV Show Crime TV Shows, International TV Shows..."
3,Jailbirds New Orleans,"TV Show Docuseries, Reality TV Feuds, flirtati..."
4,Kota Factory,"TV Show International TV Shows, Romantic TV Sh..."


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["combined"])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

In [23]:
def recommend(title, cosine_sim=cosine_sim):
    idx = indices[title]
    
    selected_type = df.loc[idx, "type"]
    selected_genre = df.loc[idx, "listed_in"]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    filtered_scores = []
    for i, score in sim_scores:
        if i == idx:
            continue
        
        # same type only
        if df.loc[i, "type"] != selected_type:
            continue
        
        # at least one common genre keyword
        if not any(g.strip() in df.loc[i, "listed_in"] for g in selected_genre.split(",")):
            continue
        
        filtered_scores.append((i, score))
        
        if len(filtered_scores) == 5:
            break

    movie_indices = [i[0] for i in filtered_scores]
    return df[["title", "type", "listed_in"]].iloc[movie_indices]

In [24]:
recommend("Stranger Things")

,title,type,listed_in
3986,The OA,TV Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy"
241,Manifest,TV Show,"TV Dramas, TV Mysteries, TV Sci-Fi & Fantasy"
4518,The Haunting of Hill House,TV Show,"TV Dramas, TV Horror, TV Mysteries"
2417,13 Reasons Why,TV Show,"Crime TV Shows, TV Dramas, TV Mysteries"
3187,Nightflyers,TV Show,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy"


In [25]:
recommendation_rows = []

for title in df["title"].dropna().unique():
    try:
        recs = recommend(title)
        for _, row in recs.iterrows():
            recommendation_rows.append({
                "selected_title": title,
                "recommended_title": row["title"],
                "type": row["type"],
                "genre": row["listed_in"]
            })
    except:
        pass

recommendations_df = pd.DataFrame(recommendation_rows)
recommendations_df.to_csv("recommendations.csv", index=False)

In [26]:
df.to_csv("netflix_cleaned_final1.csv", index=False)
print("netflix_cleaned_final1.csv saved")

netflix_cleaned_final1.csv saved


In [37]:
df.dtypes

show_id                 object
type                    object
title                   object
director                object
cast                    object
country                 object
date_added      datetime64[ns]
release_year             int64
rating                  object
duration                object
listed_in               object
description             object
year_added               int64
month_added              int64
month_name              object
combined                object
dtype: object

In [27]:
df["listed_in"] = df["listed_in"].str.split(", ")

df_genre = df.explode("listed_in")

In [30]:
df_genre.to_csv("netflix_genre.csv", index=False)